# ZuCo 2.0 EEG Data Extraction & Visualization Pipeline
This notebook provides a pure Python pipeline to read ZuCo 2.0 `.mat` files (v7.3 format), extract the EEG data, and visualize the EEG mappings at both the **sentence level** and **word level**.

### Prerequisites
Make sure you have the required libraries installed:
```bash
pip install h5py numpy matplotlib scipy
```

### Download ZuCo 2.0 Dataset
The following cell contains the download script to automatically download the ZuCo 2.0 dataset from OSF (approx 120GB). It supports resuming if interrupted.

In [1]:
import os
import sys

try:
    import requests
    from tqdm import tqdm
except ImportError:
    print("Please install required packages before running:")
    print("pip install requests tqdm")
    sys.exit(1)

# OSF Node ID for ZuCo 2.0
NODE_ID = "2urht"
BASE_URL = f"https://api.osf.io/v2/nodes/{NODE_ID}/files/osfstorage/"

# Resolve dataset directory (../dataset/zuco2)
# Resolve dataset directory (../dataset/zuco2)
PROJECT_ROOT = os.path.dirname(os.getcwd())
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "zuco2")

import time


def get_osf_data(api_url, max_retries=5):
    """Fetch folder metadata from OSF API with retries, handling pagination."""
    all_data = []
    current_url = api_url

    while current_url:
        for attempt in range(max_retries):
            try:
                response = requests.get(current_url, timeout=30)
                response.raise_for_status()
                json_data = response.json()
                all_data.extend(json_data["data"])

                # Check for next page
                links = json_data.get("links", {})
                current_url = links.get("next")
                break  # Break retry loop if successful
            except requests.exceptions.RequestException as e:
                if attempt < max_retries - 1:
                    print(
                        f"Network error while fetching metadata: {e}. Retrying in 5s... ({attempt + 1}/{max_retries})"
                    )
                    time.sleep(5)
                else:
                    raise
    return all_data


def download_file_resumable(url, destination, max_retries=5):
    """Downloads a file with resume support and retries."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    temp_destination = destination + ".tmp"

    for attempt in range(max_retries):
        file_size = 0
        if os.path.exists(destination):
            print(
                f"File {os.path.basename(destination)} already fully downloaded. Skipping."
            )
            return True

        if os.path.exists(temp_destination):
            file_size = os.path.getsize(temp_destination)

        headers = {"Range": f"bytes={file_size}-"} if file_size > 0 else {}

        try:
            response = requests.get(
                url, headers=headers, stream=True, allow_redirects=True, timeout=30
            )

            # 416 Range Not Satisfiable means we requested a range past the end of the file
            if response.status_code == 416:
                os.rename(temp_destination, destination)
                print(f"File {os.path.basename(destination)} already fully downloaded.")
                return True

            if response.status_code not in [200, 206]:
                print(f"Failed to download {url}. Status code: {response.status_code}")
                return False

            if file_size > 0 and response.status_code == 200:
                print("Server doesn't support resume. Restarting download...")
                file_size = 0
                mode = "wb"
            else:
                mode = "ab"

            total_size = int(response.headers.get("content-length", 0)) + file_size

            with (
                open(temp_destination, mode) as f,
                tqdm(
                    desc=os.path.basename(destination),
                    total=total_size,
                    initial=file_size,
                    unit="iB",
                    unit_scale=True,
                    unit_divisor=1024,
                ) as bar,
            ):
                for chunk in response.iter_content(chunk_size=8192 * 4):
                    if chunk:
                        size = f.write(chunk)
                        bar.update(size)

            # Verify the file is complete
            if total_size == 0 or os.path.getsize(temp_destination) >= total_size:
                os.rename(temp_destination, destination)
                return True
            else:
                print(f"Download incomplete for {os.path.basename(destination)}")
                # Will retry in the next loop iteration

        except Exception as e:  # noqa: BLE001, S110
            if attempt < max_retries - 1:
                print(
                    f"\nError downloading {os.path.basename(destination)}: {e}. Retrying in 5s... ({attempt + 1}/{max_retries})"
                )
                time.sleep(5)
            else:
                print(
                    f"\nFailed to download {os.path.basename(destination)} after {max_retries} attempts: {e}"
                )
                return False


def traverse_and_download(api_url, current_path):
    """Recursively traverses OSF folders and downloads files sequentially."""
    print(f"Fetching listing for {os.path.relpath(current_path, PROJECT_ROOT)} ...")
    items = get_osf_data(api_url)

    for item in items:
        kind = item["attributes"]["kind"]
        name = item["attributes"]["name"]

        if kind == "folder":
            next_url = item["relationships"]["files"]["links"]["related"]["href"]
            next_path = os.path.join(current_path, name)
            success = traverse_and_download(next_url, next_path)
            if not success:
                return False
        elif kind == "file":
            download_url = item["links"]["download"]
            file_path = os.path.join(current_path, name)

            # Download file sequentially, stopping if one fails
            success = download_file_resumable(download_url, file_path)
            if not success:
                print(f"Stopping download process because {name} failed.")
                return False

    return True


# Execute Download
print(f"Starting download of ZuCo 2.0 to: {DATASET_DIR}")
print("-" * 50)
os.makedirs(DATASET_DIR, exist_ok=True)

success = traverse_and_download(BASE_URL, DATASET_DIR)

if success:
    print("-" * 50)
    print("Dataset download completed successfully!")
else:
    print("-" * 50)
    print("Dataset download interrupted. Run the script again to resume.")

Starting download of ZuCo 2.0 to: c:\SDE Projects\Open-BCI-EEG-Waves-To-Text-Translation-And-further-Robotic-Implementaions\dataset\zuco2
--------------------------------------------------
Fetching listing for dataset\zuco2 ...
Fetching listing for dataset\zuco2\task_materials ...
File nr_7.csv already fully downloaded. Skipping.
File nr_4_control_questions.csv already fully downloaded. Skipping.
File tsr_3.csv already fully downloaded. Skipping.
File tsr_5.csv already fully downloaded. Skipping.
File tsr_2.csv already fully downloaded. Skipping.
File nr_4.csv already fully downloaded. Skipping.
File tsr_7.csv already fully downloaded. Skipping.
File nr_1_control_questions.csv already fully downloaded. Skipping.
File nr_3.csv already fully downloaded. Skipping.
File nr_1.csv already fully downloaded. Skipping.
File nr_2.csv already fully downloaded. Skipping.
File tsr_1.csv already fully downloaded. Skipping.
File nr_5.csv already fully downloaded. Skipping.
File tsr_4.csv already full

In [2]:
import glob
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np

# Setup paths
PROJECT_ROOT = os.path.dirname(os.getcwd())
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "zuco2")

# Output directories for visualizations
WORD_MAPPING_DIR = os.path.join(PROJECT_ROOT, "dataset", "word eeg mapping")
SENTENCE_MAPPING_DIR = os.path.join(PROJECT_ROOT, "dataset", "sentence eeg mapping")

os.makedirs(WORD_MAPPING_DIR, exist_ok=True)
os.makedirs(SENTENCE_MAPPING_DIR, exist_ok=True)

print(f"Data directory: {DATASET_DIR}")
print(f"Word mapping output: {WORD_MAPPING_DIR}")
print(f"Sentence mapping output: {SENTENCE_MAPPING_DIR}")

Data directory: c:\SDE Projects\Open-BCI-EEG-Waves-To-Text-Translation-And-further-Robotic-Implementaions\dataset\zuco2
Word mapping output: c:\SDE Projects\Open-BCI-EEG-Waves-To-Text-Translation-And-further-Robotic-Implementaions\dataset\word eeg mapping
Sentence mapping output: c:\SDE Projects\Open-BCI-EEG-Waves-To-Text-Translation-And-further-Robotic-Implementaions\dataset\sentence eeg mapping


### Helper Functions
Functions to extract strings from `h5py` references and plot the EEG graphs.

In [3]:
def get_string(f, ref):
    """Extracts string from h5py object reference."""
    try:
        obj = f[ref]
        return "".join(chr(c[0]) for c in obj[:])
    except Exception:  # noqa: BLE001, S110
        return "Unknown"


CHANNEL_LABELS = [
    "E2",
    "E3",
    "E4",
    "E5",
    "E6",
    "E7",
    "E9",
    "E10",
    "E11",
    "E12",
    "E13",
    "E15",
    "E16",
    "E18",
    "E19",
    "E20",
    "E22",
    "E23",
    "E24",
    "E26",
    "E27",
    "E28",
    "E29",
    "E30",
    "E31",
    "E33",
    "E34",
    "E35",
    "E36",
    "E37",
    "E38",
    "E39",
    "E40",
    "E41",
    "E42",
    "E43",
    "E44",
    "E45",
    "E46",
    "E47",
    "E50",
    "E51",
    "E52",
    "E53",
    "E54",
    "E55",
    "E57",
    "E58",
    "E59",
    "E60",
    "E61",
    "E62",
    "E64",
    "E65",
    "E66",
    "E67",
    "E69",
    "E70",
    "E71",
    "E72",
    "E74",
    "E75",
    "E76",
    "E77",
    "E78",
    "E79",
    "E80",
    "E82",
    "E83",
    "E84",
    "E85",
    "E86",
    "E87",
    "E89",
    "E90",
    "E91",
    "E92",
    "E93",
    "E95",
    "E96",
    "E97",
    "E98",
    "E100",
    "E101",
    "E102",
    "E103",
    "E104",
    "E105",
    "E106",
    "E108",
    "E109",
    "E110",
    "E111",
    "E112",
    "E114",
    "E115",
    "E116",
    "E117",
    "E118",
    "E120",
    "E121",
    "E122",
    "E123",
    "E124",
    "Cz",
]


def plot_eeg(eeg_data, title, filename, channel_labels=CHANNEL_LABELS):
    """
    Plots all EEG channels on a single heatmap graph and saves the image.
    eeg_data shape is usually (channels, time) or (time, channels).
    """
    if eeg_data is None or eeg_data.size == 0:
        return

    eeg_data = np.array(eeg_data)
    # Ensure shape is (channels, time)
    if eeg_data.shape[0] > eeg_data.shape[1]:
        eeg_data = eeg_data.T

    num_channels, _time_points = eeg_data.shape

    _, ax = plt.subplots(figsize=(15, 20))
    cax = ax.imshow(eeg_data, aspect="auto", cmap="viridis")
    fig.colorbar(cax, ax=ax, label="Amplitude")

    ax.set_title(title)
    ax.set_xlabel("Time points")
    ax.set_ylabel("EEG Channels")

    if channel_labels and num_channels == len(channel_labels):
        ax.set_yticks(np.arange(num_channels))
        ax.set_yticklabels(channel_labels, fontsize=8)
    else:
        ax.set_yticks(np.arange(0, num_channels, max(1, num_channels // 20)))

    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()

### Dataset Summary
Scans all downloaded `.mat` files and produces:
- **User count** and file overview
- **Full sentence transcript** table
- **Unique words** table across all sentences


In [ ]:
import os

import pandas as pd
from IPython.display import HTML, display

# Paths — notebook cwd is dataset_pipeline/, so go up one level
PROJECT_ROOT = os.path.dirname(os.getcwd())
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "zuco2")

mat_files = glob.glob(os.path.join(DATASET_DIR, "**", "results*.mat"), recursive=True)
print(f"Found {len(mat_files)} .mat files.")


def get_string(f, ref):  # noqa: F811
    """Extracts string from h5py object reference."""
    try:
        obj = f[ref]
        return "".join(chr(c[0]) for c in obj[:])
    except Exception:  # noqa: BLE001, S110
        return ""


STRIP_CHARS = ".,;:!?\"'()[]"

all_rows = []
unique_words = set()
subjects = set()
tasks = set()

for mat_path in sorted(mat_files):
    fname = os.path.splitext(os.path.basename(mat_path))[0]
    parts = fname.replace("results", "").split("_")
    subject = parts[-1] if len(parts) >= 2 else fname
    task = "_".join(parts[:-1]) if len(parts) >= 2 else "unknown"
    subjects.add(subject)
    tasks.add(task)

    try:
        with h5py.File(mat_path, "r") as f:
            if "sentenceData" not in f:
                continue
            sd = f["sentenceData"]
            n_sentences = sd["content"].shape[0]

            num_ch = "Unknown"
            for i in range(n_sentences):
                for key in ["rawData", "mean_t1", "mean_t2"]:
                    if key in sd:
                        eeg = f[sd[key][i, 0]][:]
                        if np.array(eeg).dtype != object:
                            num_ch = min(eeg.shape[1], eeg.shape[0])
                            break
                if num_ch != "Unknown":
                    break

            for i in range(n_sentences):
                ref = sd["content"][i, 0]
                text = get_string(f, ref)
                if not text:
                    continue
                words = text.split()
                unique_words.update(w.strip(STRIP_CHARS) for w in words)
                all_rows.append(
                    {
                        "Subject": subject,
                        "Task": task,
                        "Sent #": i + 1,
                        "Sentence": text,
                        "# Words": len(words),
                        "# Channels": num_ch,
                    }
                )
    except Exception:  # noqa: BLE001, S110
        pass

print()
print("=" * 60)
print(f"  Subjects (users): {len(subjects)}")
print(f"  Tasks found:      {sorted(tasks)}")
print(f"  Total sentences:  {len(all_rows)}")
print(f"  Unique words:     {len(unique_words)}")
print("=" * 60)

df = pd.DataFrame(all_rows)

### Users & Tasks Overview
Summary table of each subject and how many sentences they contributed per task, along with the number of EEG channels.


In [ ]:
if not df.empty:
    summary = (
        df.groupby(["Subject", "Task"])
        .agg(
            Sentences=("Sent #", "count"),
            Total_Words=("# Words", "sum"),
            Channels=("# Channels", "first"),
        )
        .reset_index()
    )
    print(f"Total subjects: {df['Subject'].nunique()}")
    display(HTML(summary.to_html(index=False, border=0)))

### Full Sentence Transcript
Every sentence extracted from all subjects and tasks.


In [ ]:
if not df.empty:
    display(HTML(df.to_html(index=False, border=0)))
else:
    print("No data found — run the download cell first.")

### Unique Words in Dataset
All unique tokens found across the entire corpus (punctuation stripped).


In [ ]:
if unique_words:
    sorted_words = sorted(w for w in unique_words if w)
    cols = 6
    rows_w = [sorted_words[i : i + cols] for i in range(0, len(sorted_words), cols)]
    while len(rows_w[-1]) < cols:
        rows_w[-1].append("")
    df_words = pd.DataFrame(rows_w, columns=[f"Word {j + 1}" for j in range(cols)])
    print(f"Total unique words: {len(sorted_words)}")
    display(HTML(df_words.to_html(index=False, border=0)))

### EEG Waveform Preview
Sample **sentence-level** and **word-level** EEG heatmaps from the first available `.mat` file,
with all 105 electrode names labelled on the Y-axis.


In [ ]:
CHANNEL_LABELS = [
    "E2",
    "E3",
    "E4",
    "E5",
    "E6",
    "E7",
    "E9",
    "E10",
    "E11",
    "E12",
    "E13",
    "E15",
    "E16",
    "E18",
    "E19",
    "E20",
    "E22",
    "E23",
    "E24",
    "E26",
    "E27",
    "E28",
    "E29",
    "E30",
    "E31",
    "E33",
    "E34",
    "E35",
    "E36",
    "E37",
    "E38",
    "E39",
    "E40",
    "E41",
    "E42",
    "E43",
    "E44",
    "E45",
    "E46",
    "E47",
    "E50",
    "E51",
    "E52",
    "E53",
    "E54",
    "E55",
    "E57",
    "E58",
    "E59",
    "E60",
    "E61",
    "E62",
    "E64",
    "E65",
    "E66",
    "E67",
    "E69",
    "E70",
    "E71",
    "E72",
    "E74",
    "E75",
    "E76",
    "E77",
    "E78",
    "E79",
    "E80",
    "E82",
    "E83",
    "E84",
    "E85",
    "E86",
    "E87",
    "E89",
    "E90",
    "E91",
    "E92",
    "E93",
    "E95",
    "E96",
    "E97",
    "E98",
    "E100",
    "E101",
    "E102",
    "E103",
    "E104",
    "E105",
    "E106",
    "E108",
    "E109",
    "E110",
    "E111",
    "E112",
    "E114",
    "E115",
    "E116",
    "E117",
    "E118",
    "E120",
    "E121",
    "E122",
    "E123",
    "E124",
    "Cz",
]


def plot_eeg_ax(ax, eeg_data, title):
    eeg_data = np.array(eeg_data, dtype=float)
    if len(eeg_data.shape) < 2:
        return
    if eeg_data.shape[0] > eeg_data.shape[1]:
        eeg_data = eeg_data.T
    num_ch = eeg_data.shape[0]
    im = ax.imshow(eeg_data, aspect="auto", cmap="viridis")
    plt.colorbar(im, ax=ax, label="Amplitude", fraction=0.03)
    ax.set_title(title, fontsize=9, pad=4)
    ax.set_xlabel("Time points", fontsize=8)
    ax.set_ylabel("EEG Channels", fontsize=8)
    if num_ch == len(CHANNEL_LABELS):
        ax.set_yticks(np.arange(num_ch))
        ax.set_yticklabels(CHANNEL_LABELS, fontsize=5)
    else:
        ax.set_yticks(np.arange(0, num_ch, max(1, num_ch // 20)))


# Pick the first valid HDF5 .mat file
preview_mat = None
for mat_path in sorted(mat_files):
    try:
        with h5py.File(mat_path, "r") as f:
            if "sentenceData" in f:
                preview_mat = mat_path
                break
    except Exception:  # noqa: BLE001, S110
        pass

if not preview_mat:
    print("No valid .mat file found. Run the download cell first.")
else:
    print(f"Previewing: {os.path.basename(preview_mat)}")
    N_PREVIEW = 3
    with h5py.File(preview_mat, "r") as f:
        sd = f["sentenceData"]
        n_sents = sd["content"].shape[0]

        # Sentence previews
        fig, axes = plt.subplots(1, N_PREVIEW, figsize=(18, 22))
        fig.suptitle(
            "Sentence-Level EEG Mappings", fontsize=14, fontweight="bold", y=1.01
        )
        plotted = 0
        for i in range(n_sents):
            if plotted >= N_PREVIEW:
                break
            sent_text = get_string(f, sd["content"][i, 0])
            eeg_sent = None
            for key in ["rawData", "mean_t1"]:
                if key in sd:
                    eeg_sent = f[sd[key][i, 0]][:]
                    break
            if eeg_sent is None or np.array(eeg_sent).dtype == object:
                continue
            plot_eeg_ax(axes[plotted], eeg_sent, f"Sent {i + 1}: {sent_text[:30]}...")
            plotted += 1
        plt.tight_layout()
        plt.show()

        # Word previews
        fig, axes = plt.subplots(1, N_PREVIEW, figsize=(18, 22))
        fig.suptitle(
            "Word-Level EEG Mappings (1st Fixation)",
            fontsize=14,
            fontweight="bold",
            y=1.01,
        )
        plotted = 0
        for i in range(n_sents):
            if plotted >= N_PREVIEW:
                break
            if "word" not in sd:
                break
            words_grp = f[sd["word"][i, 0]]
            for w in range(words_grp["content"].shape[0]):
                if plotted >= N_PREVIEW:
                    break
                w_text = get_string(f, words_grp["content"][w, 0])
                eeg_word = None
                for key in ["rawEEG", "FFD_t1"]:
                    if key in words_grp:
                        raw = f[words_grp[key][w, 0]][:]
                        if raw.dtype == object:
                            raw = f[raw[0, 0]][:]
                        if len(raw.shape) == 2:
                            eeg_word = raw
                        break
                if eeg_word is None:
                    continue
                plot_eeg_ax(axes[plotted], eeg_word, f"Word: '{w_text}'")
                plotted += 1
        plt.tight_layout()
        plt.show()

### Save All EEG Visualizations to Disk
Generates and saves labelled EEG heatmaps for all sentences and one word per sentence
into `dataset/sentence eeg mapping/` and `dataset/word eeg mapping/`.


In [ ]:
SENT_MAPPING_DIR = os.path.join(PROJECT_ROOT, "dataset", "sentence eeg mapping")
WORD_MAPPING_DIR = os.path.join(PROJECT_ROOT, "dataset", "word eeg mapping")
os.makedirs(SENT_MAPPING_DIR, exist_ok=True)
os.makedirs(WORD_MAPPING_DIR, exist_ok=True)


def save_eeg_plot(eeg_data, title, filepath):
    eeg_data = np.array(eeg_data, dtype=float)
    if len(eeg_data.shape) < 2:
        return False
    if eeg_data.shape[0] > eeg_data.shape[1]:
        eeg_data = eeg_data.T
    num_ch = eeg_data.shape[0]
    _, ax = plt.subplots(figsize=(15, 20))
    im = ax.imshow(eeg_data, aspect="auto", cmap="viridis")
    plt.colorbar(im, ax=ax, label="Amplitude")
    ax.set_title(title)
    ax.set_xlabel("Time points")
    ax.set_ylabel("EEG Channels")
    if num_ch == len(CHANNEL_LABELS):
        ax.set_yticks(np.arange(num_ch))
        ax.set_yticklabels(CHANNEL_LABELS, fontsize=8)
    else:
        ax.set_yticks(np.arange(0, num_ch, max(1, num_ch // 20)))
    plt.tight_layout()
    plt.savefig(filepath, dpi=150)
    plt.close()
    return True


total_sent = 0
total_word = 0

for mat_path in sorted(mat_files):
    fname = os.path.splitext(os.path.basename(mat_path))[0]
    try:
        with h5py.File(mat_path, "r") as f:
            if "sentenceData" not in f:
                continue
            sd = f["sentenceData"]
            n_sents = sd["content"].shape[0]
            for i in range(n_sents):
                sent_text = get_string(f, sd["content"][i, 0])
                safe = "".join(c if c.isalnum() else "_" for c in sent_text)[:50]
                # Sentence
                for key in ["rawData", "mean_t1"]:
                    if key in sd:
                        eeg = f[sd[key][i, 0]][:]
                        if np.array(eeg).dtype != object:
                            out = os.path.join(
                                SENT_MAPPING_DIR, f"{fname}_sent_{i}_{safe}.png"
                            )
                            if save_eeg_plot(
                                eeg, f"Sentence EEG: {sent_text[:60]}", out
                            ):
                                total_sent += 1
                        break
                # Word (first valid per sentence)
                if "word" not in sd:
                    continue
                words_grp = f[sd["word"][i, 0]]
                for w in range(words_grp["content"].shape[0]):
                    w_text = get_string(f, words_grp["content"][w, 0])
                    safe_w = "".join(c if c.isalnum() else "_" for c in w_text)
                    eeg_word = None
                    for key in ["rawEEG", "FFD_t1"]:
                        if key in words_grp:
                            raw = f[words_grp[key][w, 0]][:]
                            if raw.dtype == object:
                                raw = f[raw[0, 0]][:]
                            if len(raw.shape) == 2:
                                eeg_word = raw
                            break
                    if eeg_word is not None:
                        out = os.path.join(
                            WORD_MAPPING_DIR, f"{fname}_sent_{i}_word_{w}_{safe_w}.png"
                        )
                        if save_eeg_plot(eeg_word, f"Word EEG: {w_text}", out):
                            total_word += 1
                        break
    except Exception:  # noqa: BLE001, S110
        print(f"  Skipping {fname}")

print(f"Done! Saved {total_sent} sentence EEG images and {total_word} word EEG images.")

### Save Data to HDF5 Files (Person-Centric Approach)
Implements the data storage process organized by subject:
- **Person-centric Cache:** `dataset/extracted/person/<subject>/<task>/<transcript>.h5` (Identical zero-data-loss arrays)
- **Format & Quality:** Native `dtype` (typically `float64`) + lossless `gzip` compression.
- **Expected Space:** ~1.5-2 GB total.
- **Speed:** Extremely fast disk I/O, optimized for chunked DataLoader reads in PyTorch/TF.

In [ ]:
import glob
import os

import h5py
import numpy as np

PROJECT_ROOT = os.path.dirname(os.getcwd())
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "zuco2")
EXTRACTED_DIR = os.path.join(PROJECT_ROOT, "dataset", "extracted")
PERSON_DIR = os.path.join(EXTRACTED_DIR, "person")
os.makedirs(PERSON_DIR, exist_ok=True)

CHANNEL_LABELS = [
    "E2",
    "E3",
    "E4",
    "E5",
    "E6",
    "E7",
    "E9",
    "E10",
    "E11",
    "E12",
    "E13",
    "E15",
    "E16",
    "E18",
    "E19",
    "E20",
    "E22",
    "E23",
    "E24",
    "E26",
    "E27",
    "E28",
    "E29",
    "E30",
    "E31",
    "E33",
    "E34",
    "E35",
    "E36",
    "E37",
    "E38",
    "E39",
    "E40",
    "E41",
    "E42",
    "E43",
    "E44",
    "E45",
    "E46",
    "E47",
    "E50",
    "E51",
    "E52",
    "E53",
    "E54",
    "E55",
    "E57",
    "E58",
    "E59",
    "E60",
    "E61",
    "E62",
    "E64",
    "E65",
    "E66",
    "E67",
    "E69",
    "E70",
    "E71",
    "E72",
    "E74",
    "E75",
    "E76",
    "E77",
    "E78",
    "E79",
    "E80",
    "E82",
    "E83",
    "E84",
    "E85",
    "E86",
    "E87",
    "E89",
    "E90",
    "E91",
    "E92",
    "E93",
    "E95",
    "E96",
    "E97",
    "E98",
    "E100",
    "E101",
    "E102",
    "E103",
    "E104",
    "E105",
    "E106",
    "E108",
    "E109",
    "E110",
    "E111",
    "E112",
    "E114",
    "E115",
    "E116",
    "E117",
    "E118",
    "E120",
    "E121",
    "E122",
    "E123",
    "E124",
    "Cz",
]


def get_string(f, ref):
    try:
        obj = f[ref]
        return "".join(chr(c[0]) for c in obj[:])
    except Exception:  # noqa: BLE001, S110
        return "Unknown"


mat_files = glob.glob(os.path.join(DATASET_DIR, "**", "results*.mat"), recursive=True)
valid_files = [
    f for f in mat_files if os.path.getsize(f) > 1000000
]  # Skip tiny v7 metadata files

saved_unified = 0

for mat_path in sorted(valid_files):  # Process all valid files
    fname = os.path.splitext(os.path.basename(mat_path))[0]
    parts = fname.replace("results", "").split("_")
    subject = parts[0] if len(parts) >= 2 else fname
    task = parts[1] if len(parts) >= 2 else "Unknown"
    subject_dir = os.path.join(PERSON_DIR, subject, task)
    os.makedirs(subject_dir, exist_ok=True)

    try:
        with h5py.File(mat_path, "r") as f:
            if "sentenceData" not in f:
                continue
            sd = f["sentenceData"]
            n_sents = sd["content"].shape[0]
            for i in range(n_sents):
                sent_text = get_string(f, sd["content"][i, 0])
                transcript_safe = "".join(c if c.isalnum() else "_" for c in sent_text)[
                    :50
                ]
                if not transcript_safe:
                    transcript_safe = f"sent_{i}"

                eeg_sent = None
                for key in ["rawData", "mean_t1", "mean_t2"]:
                    if key in sd:
                        eeg_sent = f[sd[key][i, 0]][:]
                        if np.array(eeg_sent).dtype != object:
                            break

                if eeg_sent is None or np.array(eeg_sent).dtype == object:
                    continue

                # Ensure shape is [channels, time_steps]
                if eeg_sent.shape[0] > eeg_sent.shape[1]:
                    eeg_sent = eeg_sent.T

                # Save to person-centric cache
                unified_path = os.path.join(subject_dir, f"{transcript_safe}.h5")
                with h5py.File(unified_path, "w") as out_f:
                    out_f.create_dataset("eeg", data=eeg_sent, compression="gzip")

                saved_unified += 1

    except Exception as e:  # noqa: BLE001, S110
        print(f"Error processing {fname}: {e}")

print("Unified Matrix Data extraction complete.")
print(f"Saved {saved_unified} Unified Matrix files (.h5).")

### Save Data to HDF5 Files (Data-Centric Approach with Metadata)
Implements the data storage process with embedded ML metadata attributes:
- **Data-centric Cache:** `dataset/extracted/data/<subject>_<transcript>.h5`
- **Format & Quality:** Native `dtype` (typically `float64`) + lossless `gzip` compression.
- **Expected Space:** ~1.5-2 GB total.
- **Metadata included:** `subject`, `task`, `transcript`, `sampling_rate`, `channel_labels`.

In [ ]:
import glob
import json
import os
import sys

import h5py
import numpy as np

PROJECT_ROOT = os.path.dirname(os.getcwd())
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "zuco2")
EXTRACTED_DIR = os.path.join(PROJECT_ROOT, "dataset", "extracted")
DATA_DIR = os.path.join(EXTRACTED_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)

CHANNEL_LABELS = [
    "E2",
    "E3",
    "E4",
    "E5",
    "E6",
    "E7",
    "E9",
    "E10",
    "E11",
    "E12",
    "E13",
    "E15",
    "E16",
    "E18",
    "E19",
    "E20",
    "E22",
    "E23",
    "E24",
    "E26",
    "E27",
    "E28",
    "E29",
    "E30",
    "E31",
    "E33",
    "E34",
    "E35",
    "E36",
    "E37",
    "E38",
    "E39",
    "E40",
    "E41",
    "E42",
    "E43",
    "E44",
    "E45",
    "E46",
    "E47",
    "E50",
    "E51",
    "E52",
    "E53",
    "E54",
    "E55",
    "E57",
    "E58",
    "E59",
    "E60",
    "E61",
    "E62",
    "E64",
    "E65",
    "E66",
    "E67",
    "E69",
    "E70",
    "E71",
    "E72",
    "E74",
    "E75",
    "E76",
    "E77",
    "E78",
    "E79",
    "E80",
    "E82",
    "E83",
    "E84",
    "E85",
    "E86",
    "E87",
    "E89",
    "E90",
    "E91",
    "E92",
    "E93",
    "E95",
    "E96",
    "E97",
    "E98",
    "E100",
    "E101",
    "E102",
    "E103",
    "E104",
    "E105",
    "E106",
    "E108",
    "E109",
    "E110",
    "E111",
    "E112",
    "E114",
    "E115",
    "E116",
    "E117",
    "E118",
    "E120",
    "E121",
    "E122",
    "E123",
    "E124",
    "Cz",
]


def get_string(f, ref):
    try:
        obj = f[ref]
        return "".join(chr(c[0]) for c in obj[:])
    except Exception:  # noqa: BLE001, S110
        return "Unknown"


mat_files = glob.glob(os.path.join(DATASET_DIR, "**", "results*.mat"), recursive=True)
valid_files = [
    f for f in mat_files if os.path.getsize(f) > 1000000
]  # Skip tiny v7 metadata files

saved_unified = 0

for mat_path in sorted(valid_files):  # Process all valid files
    fname = os.path.splitext(os.path.basename(mat_path))[0]
    parts = fname.replace("results", "").split("_")
    subject = parts[0] if len(parts) >= 2 else fname
    task = parts[1] if len(parts) >= 2 else "Unknown"

    try:
        with h5py.File(mat_path, "r") as f:
            if "sentenceData" not in f:
                continue
            sd = f["sentenceData"]
            n_sents = sd["content"].shape[0]
            for i in range(n_sents):
                sent_text = get_string(f, sd["content"][i, 0])
                transcript_safe = "".join(c if c.isalnum() else "_" for c in sent_text)[
                    :50
                ]
                if not transcript_safe:
                    transcript_safe = f"sent_{i}"

                eeg_sent = None
                for key in ["rawData", "mean_t1", "mean_t2"]:
                    if key in sd:
                        eeg_sent = f[sd[key][i, 0]][:]
                        if np.array(eeg_sent).dtype != object:
                            break

                if eeg_sent is None or np.array(eeg_sent).dtype == object:
                    continue

                # Ensure shape is [channels, time_steps]
                if eeg_sent.shape[0] > eeg_sent.shape[1]:
                    eeg_sent = eeg_sent.T

                # Save to data-centric cache with metadata
                data_path = os.path.join(DATA_DIR, f"{subject}_{transcript_safe}.h5")
                with h5py.File(data_path, "w") as out_f:
                    dset = out_f.create_dataset(
                        "eeg", data=eeg_sent, compression="gzip"
                    )
                    # Attach ML metadata
                    out_f.attrs["subject"] = subject
                    out_f.attrs["task"] = task
                    out_f.attrs["transcript"] = sent_text
                    out_f.attrs["sampling_rate"] = 500
                    out_f.attrs["channel_labels"] = (
                        json.dumps(CHANNEL_LABELS)
                        if "json" in sys.modules
                        else str(CHANNEL_LABELS)
                    )

                saved_unified += 1

    except Exception as e:  # noqa: BLE001, S110
        print(f"Error processing {fname}: {e}")

print("Unified Matrix Data extraction complete.")
print(f"Saved {saved_unified} Unified Matrix files (.h5).")

### Save Data to .npy Files (Split by Node Approach)
Implements the **Not Recommended** data storage process (for comparison/legacy requirements):
- Split by Node Approach: `dataset/extracted/person/<subject>/<task>/<transcript>/<node_name>.npy`
- **Expected Space:** ~4-6 GB total. Causes massive OS block space waste due to millions of tiny files.
- **Speed:** Very slow random seek bottlenecks.

In [ ]:
import glob
import os

import h5py
import numpy as np

PROJECT_ROOT = os.path.dirname(os.getcwd())
DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset", "zuco2")
EXTRACTED_DIR = os.path.join(PROJECT_ROOT, "dataset", "extracted")
PERSON_DIR = os.path.join(EXTRACTED_DIR, "person")
os.makedirs(PERSON_DIR, exist_ok=True)


def get_string(f, ref):
    try:
        obj = f[ref]
        return "".join(chr(c[0]) for c in obj[:])
    except Exception:  # noqa: BLE001, S110
        return "Unknown"


CHANNEL_LABELS = [
    "E2",
    "E3",
    "E4",
    "E5",
    "E6",
    "E7",
    "E9",
    "E10",
    "E11",
    "E12",
    "E13",
    "E15",
    "E16",
    "E18",
    "E19",
    "E20",
    "E22",
    "E23",
    "E24",
    "E26",
    "E27",
    "E28",
    "E29",
    "E30",
    "E31",
    "E33",
    "E34",
    "E35",
    "E36",
    "E37",
    "E38",
    "E39",
    "E40",
    "E41",
    "E42",
    "E43",
    "E44",
    "E45",
    "E46",
    "E47",
    "E50",
    "E51",
    "E52",
    "E53",
    "E54",
    "E55",
    "E57",
    "E58",
    "E59",
    "E60",
    "E61",
    "E62",
    "E64",
    "E65",
    "E66",
    "E67",
    "E69",
    "E70",
    "E71",
    "E72",
    "E74",
    "E75",
    "E76",
    "E77",
    "E78",
    "E79",
    "E80",
    "E82",
    "E83",
    "E84",
    "E85",
    "E86",
    "E87",
    "E89",
    "E90",
    "E91",
    "E92",
    "E93",
    "E95",
    "E96",
    "E97",
    "E98",
    "E100",
    "E101",
    "E102",
    "E103",
    "E104",
    "E105",
    "E106",
    "E108",
    "E109",
    "E110",
    "E111",
    "E112",
    "E114",
    "E115",
    "E116",
    "E117",
    "E118",
    "E120",
    "E121",
    "E122",
    "E123",
    "E124",
    "Cz",
]

mat_files = glob.glob(os.path.join(DATASET_DIR, "**", "results*.mat"), recursive=True)
valid_files = [
    f for f in mat_files if os.path.getsize(f) > 1000000
]  # Skip tiny v7 metadata files

saved_split = 0

for mat_path in sorted(valid_files):  # Process all valid files
    fname = os.path.splitext(os.path.basename(mat_path))[0]
    parts = fname.replace("results", "").split("_")
    subject = parts[0] if len(parts) >= 2 else fname
    task = parts[1] if len(parts) >= 2 else "Unknown"
    subject_dir = os.path.join(PERSON_DIR, subject, task)
    os.makedirs(subject_dir, exist_ok=True)

    try:
        with h5py.File(mat_path, "r") as f:
            if "sentenceData" not in f:
                continue
            sd = f["sentenceData"]
            n_sents = sd["content"].shape[0]
            for i in range(n_sents):
                sent_text = get_string(f, sd["content"][i, 0])
                transcript_safe = "".join(c if c.isalnum() else "_" for c in sent_text)[
                    :50
                ]
                if not transcript_safe:
                    transcript_safe = f"sent_{i}"

                eeg_sent = None
                for key in ["rawData", "mean_t1", "mean_t2"]:
                    if key in sd:
                        eeg_sent = f[sd[key][i, 0]][:]
                        if np.array(eeg_sent).dtype != object:
                            break

                if eeg_sent is None or np.array(eeg_sent).dtype == object:
                    continue

                # Ensure shape is [channels, time_steps]
                if eeg_sent.shape[0] > eeg_sent.shape[1]:
                    eeg_sent = eeg_sent.T

                num_ch = eeg_sent.shape[0]

                split_dir = os.path.join(subject_dir, transcript_safe)
                os.makedirs(split_dir, exist_ok=True)

                # Use available channels up to length of labels
                for ch_idx in range(num_ch):
                    ch_name = (
                        CHANNEL_LABELS[ch_idx]
                        if ch_idx < len(CHANNEL_LABELS)
                        else f"CH{ch_idx}"
                    )
                    node_path = os.path.join(split_dir, f"{ch_name}.npy")
                    np.save(node_path, eeg_sent[ch_idx, :])
                saved_split += 1

    except Exception as e:  # noqa: BLE001, S110
        print(f"Error processing {fname}: {e}")

print("Split by Node Data extraction complete.")
print(
    f"Saved {saved_split} Split by Node directories (each containing {len(CHANNEL_LABELS)} files)."
)